# 🚀 CinemaBox 1-Click Heavyweight & Multi-Modal AI Server
### 5 TB Google Storage + Colab GPU + Cloudflare Tunnel + Claude Code Agent Protocol

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/princesharmaofficial1-hub/cinemabox-colab/blob/main/deploy_70b_google_storage.ipynb)

---

## ⚡ 1-CLICK MASTER RUNNER (Press Play Below)
Select your model from the dropdown, then click the **Play** button on the cell below.
It will automatically mount your **5 TB Google Drive**, prepare the GPU engine, launch the **Web Agent Studio**, and give you a global HTTPS endpoint to use **anywhere** (CinemaBox app, VS Code, Web, Python, cURL).

In [ ]:
#@title 🚀 1-Click Launch Everything (5 TB Drive + GPU + Cloudflare) { vertical-output: true, form-width: "100%" }
Model_Choice = "Qwen 2.5 72B (World #1 Open Dense Reasoning)" #@param ["Qwen 2.5 72B (World #1 Open Dense Reasoning)", "Mixtral 8x22B (141B Heavyweight MoE)", "Qwen 2 VL 72B (Full Multi-Modal Vision + Reasoning)", "Qwen 2.5 Coder 32B (Claude Code Specialist)", "LLaMA 3.3 70B Instruct"]
Context_Window = 8192 #@param [4096, 8192, 16384, 32768] {type:"integer"}
API_Key = "sk-cinemabox70b" #@param {type:"string"}

import os
import sys
import time
import subprocess
import re

print("="*75)
print("🚀 STARTING CINEMABOX 1-CLICK AUTONOMOUS SETUP...")
print("="*75)

# -----------------------------------------------------------------------
# 1. Check GPU
# -----------------------------------------------------------------------
print("\n[1/5] 🎮 Checking GPU Acceleration...")
gpu_check = subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
print(f"      Detected: {gpu_check}")

# -----------------------------------------------------------------------
# 2. Mount 5 TB Google Drive
# -----------------------------------------------------------------------
print("\n[2/5] 📁 Mounting your 5 TB Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/CinemaBox_Models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print(f"      ✅ Google Drive Connected! Permanent Storage: {DRIVE_MODEL_DIR}")

# -----------------------------------------------------------------------
# 3. Install Pre-compiled CUDA Engine & Cloudflare Tunnel
# -----------------------------------------------------------------------
print("\n[3/5] ⚡ Installing GPU Engine & Cloudflare Tunnel...")
subprocess.run(["pip", "install", "--upgrade", "pip", "-q"], check=True)
subprocess.run([
    "pip", "install", "llama-cpp-python",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu122",
    "-q"
], check=True)
subprocess.run([
    "pip", "install", "llama-cpp-python[server]", "fastapi", "uvicorn", "huggingface_hub", "requests", "pydantic", "pillow", "-q"
], check=True)

# Install cloudflared
subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("      ✅ GPU Engine & Cloudflare Tunnel ready!")

# -----------------------------------------------------------------------
# 4. Resolve & Verify Selected Model
# -----------------------------------------------------------------------
print(f"\n[4/5] 📦 Loading Model: {Model_Choice}...")
from huggingface_hub import hf_hub_download

MODEL_MAP = {
    "Qwen 2.5 72B (World #1 Open Dense Reasoning)": {
        "repo": "bartowski/Qwen2.5-72B-Instruct-GGUF",
        "file": "Qwen2.5-72B-Instruct-Q4_K_M.gguf",
        "id": "Qwen2.5-72B-Instruct"
    },
    "Mixtral 8x22B (141B Heavyweight MoE)": {
        "repo": "bartowski/Mixtral-8x22B-Instruct-v0.1-GGUF",
        "file": "Mixtral-8x22B-Instruct-v0.1-Q4_K_M.gguf",
        "id": "Mixtral-8x22B-Instruct"
    },
    "Qwen 2 VL 72B (Full Multi-Modal Vision + Reasoning)": {
        "repo": "bartowski/Qwen2-VL-72B-Instruct-GGUF",
        "file": "Qwen2-VL-72B-Instruct-Q4_K_M.gguf",
        "id": "Qwen2-VL-72B-Instruct"
    },
    "Qwen 2.5 Coder 32B (Claude Code Specialist)": {
        "repo": "bartowski/Qwen2.5-Coder-32B-Instruct-GGUF",
        "file": "Qwen2.5-Coder-32B-Instruct-Q4_K_M.gguf",
        "id": "Qwen2.5-Coder-32B-Instruct"
    },
    "LLaMA 3.3 70B Instruct": {
        "repo": "bartowski/Llama-3.3-70B-Instruct-GGUF",
        "file": "Llama-3.3-70B-Instruct-Q4_K_M.gguf",
        "id": "Llama-3.3-70B-Instruct"
    }
}

selected = MODEL_MAP[Model_Choice]
model_path = os.path.join(DRIVE_MODEL_DIR, selected["file"])

if os.path.exists(model_path) and os.path.getsize(model_path) > 1024*1024*100:
    print(f"      🎯 Reusing cached model in 5 TB Google Drive: {selected['file']}")
    print(f"         Size: {os.path.getsize(model_path)/(1024**3):.2f} GB")
else:
    print(f"      📥 Downloading {selected['file']} directly into your 5 TB Google Drive...")
    model_path = hf_hub_download(
        repo_id=selected["repo"],
        filename=selected["file"],
        local_dir=DRIVE_MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f"      ✅ Saved permanently to: {model_path}")

# -----------------------------------------------------------------------
# 5. Launch Inference Server & Cloudflare Public HTTPS Tunnel
# -----------------------------------------------------------------------
print("\n[5/5] 🌐 Starting OpenAI-Compatible Server & Cloudflare Tunnel...")
server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", model_path,
    "--n_gpu_layers", "35",
    "--n_ctx", str(Context_Window),
    "--host", "127.0.0.1",
    "--port", "8000",
    "--api_key", API_Key,
    "--chat_format", "chatml"
]

server_proc = subprocess.Popen(server_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(10) # Initialize neural layers in CUDA

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cf_proc.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*75)
    print("🎉 EVERYTHING IS ONLINE AND READY TO USE ANYWHERE!")
    print("="*75)
    print(f"\n👉 🎨 Web Agent Studio:         {public_url}")
    print(f"👉 📱 CinemaBox App Endpoint:     {public_url}/v1/chat/completions")
    print(f"👉 💻 VS Code (Roo/Continue) Base: {public_url}/v1")
    print(f"👉 🔑 API Key:                     {API_Key}")
    print(f"👉 🧠 Active Model:                {selected['id']}")
    print("\n" + "-"*75)
    print("📱 HOW TO USE IN CINEMABOX APP:")
    print(f"   1. Open CinemaBox on phone > Settings > On-Device Intelligence")
    print(f"   2. Tap '141B MoE / 72B Multi-Modal' tab")
    print(f"   3. Paste '{public_url}/v1/chat/completions' into Host API Endpoint")
    print(f"   4. Tap Save & Run Test -> You are connected!")
    print("\n💻 HOW TO USE IN VS CODE (Roo Code / Continue / Cline):")
    print(f"   Provider: OpenAI Compatible")
    print(f"   Base URL: {public_url}/v1")
    print(f"   API Key:  {API_Key}")
    print(f"   Model:    {selected['id']}")
    print("\n🌐 HOW TO USE IN PYTHON / ANYWHERE:")
    print(f"   client = OpenAI(base_url='{public_url}/v1', api_key='{API_Key}')")
    print("="*75 + "\n")
else:
    print("❌ Cloudflare tunnel could not be resolved. Please re-run the cell.")

--- 
### Step 2 (Optional): Quick Verification Test
Run this cell to test your live model directly from Colab.

In [ ]:
import requests

prompt = "Act as an elite film curator. Recommend 3 mind-bending sci-fi movies for someone who loved Arrival and Blade Runner 2049. Include a 1-sentence premise for each."

print(f"💬 Sending prompt: '{prompt}'\n")
res = requests.post(
    f"{public_url}/v1/chat/completions",
    headers={"Authorization": f"Bearer {API_Key}"},
    json={
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 350,
        "temperature": 0.7
    }
)

if res.status_code == 200:
    print("🎬 Model Output:\n")
    print(res.json()["choices"][0]["message"]["content"])
else:
    print(f"Error {res.status_code}: {res.text}")